In [ ]:
# Optional Colab mount: uncomment if running on Colab.
# from google.colab import drive
# drive.mount("/content/drive")

import os
import subprocess

WORK_DIR = os.environ.get("WORK_DIR", os.getcwd())
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

WORK_DIR = os.path.abspath(".")
MASKSDM_DIR = os.path.abspath("../MaskSDM-MEE")
DATA_DIR = WORK_DIR
CKPT_DIR = WORK_DIR
RESULTS_DIR = os.path.join(WORK_DIR, "results_masksdm_butterfly_1000_epochs_seed1337")

os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(MASKSDM_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/zbirobin/MaskSDM-MEE", MASKSDM_DIR],
        check=True,
    )
    print("Repo cloned to", MASKSDM_DIR)
else:
    print("Repo already present, skipping.")

!pip install verde schedulefree elapid torcheval -q

os.chdir(MASKSDM_DIR)

print("\nSetup complete.")
print(f"  cwd:          {os.getcwd()}")
print(f"  MaskSDM repo: {MASKSDM_DIR}")
print(f"  work/data:    {DATA_DIR}")
print(f"  ckpts:        {CKPT_DIR}")
print(f"  results:      {RESULTS_DIR}")

Mounted at /content/drive
Repo already present, skipping.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.6/188.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.5/436.5 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 83.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.

Setup complete.
  cwd:          /content/drive/MyDrive/CISO/MaskSDM-MEE
  MaskSDM repo: /content/drive/MyDrive/CISO/MaskSDM-MEE
  work/data:    /content/drive/MyDrive/C

In [ ]:
import sys
sys.path.insert(0, MASKSDM_DIR)

import numpy as np
import pandas as pd
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from data_helpers import get_torch_dataset
from modules import get_model
from training_helpers import seed_everything, train

!pip install --upgrade --force-reinstall "scikit-learn>=1.6" -q
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 76.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
elapid 1.0.3 requires scikit-learn<1.6,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
Device: cuda
Imports OK


## 1. Load Your Data

In [ ]:
# ── Identical to CISO butterfly notebook ──────────────────────────────────────
DATA_CSV    = f'{DATA_DIR}/ebutterfly_na_2011_2025.csv'
SPLITS_JSON = f'{DATA_DIR}/ebutterfly_splits.json'

for f in [DATA_CSV, SPLITS_JSON]:
    print('OK  ' if os.path.exists(f) else 'MISSING  ', f)

butterflies = pd.read_csv(DATA_CSV)
butterflies = butterflies.reset_index(drop=True)

with open(SPLITS_JSON) as f:
    butterfly_splits = json.load(f)

print(f'\nLoaded {len(butterflies):,} rows x {len(butterflies.columns):,} columns')
print(f'Split keys: {list(butterfly_splits.keys())}')
butterflies.head(2)


OK   /content/drive/MyDrive/CISO/eButterfly/ebutterfly_na_2011_2025.csv
OK   /content/drive/MyDrive/CISO/eButterfly/ebutterfly_splits.json

Loaded 17,077 rows x 191 columns
Split keys: ['num_rows', 'meta', 'train', 'val', 'test']


,time,latitude,longitude,env_t2m_min,env_t2m_max,env_t2m_mean,env_tp_sum,env_ndvi,env_evi,env_soil_bdod,...,Tharsalea hyllus,Thymelicus lineola,Urbanus dorantes,Urbanus procne,Urbanus proteus,Vanessa atalanta,Vanessa cardui,Vanessa virginiensis,Vernia verna,Zerene cesonia
0,2025-05-16,44.481715,-77.682266,286.97748,299.25006,291.72070,5.594984e-03,0.6052,0.3350,118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-04-19,34.046186,-118.275999,286.35144,299.09650,292.28778,5.420297e-07,0.1364,0.0748,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# ── Identical column groups to CISO butterfly notebook ───────────────────────
env_cols     = [c for c in butterflies.columns if c.startswith('env_')]
species_cols = [c for c in butterflies.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

# eButterfly daily set has no bioclim. Group env into:
#   - "worldclim_cols" (variable name kept for downstream compatibility):
#       ERA5 daily t2m/tp + MODIS NDVI/EVI  (dynamic, 6 cols)
#   - "soilgrid_cols": SoilGrids + DEM      (static, 9 cols)
soilgrid_cols  = [c for c in env_cols if c.startswith('env_soil') or c == 'env_dem']
worldclim_cols = [c for c in env_cols if c not in soilgrid_cols]

print(f'Climate/dynamic cols ({len(worldclim_cols)}): {worldclim_cols}')
print(f'Static cols          ({len(soilgrid_cols)}):  {soilgrid_cols}')
print(f'Species cols         ({len(species_cols)}):   first 5 = {species_cols[:5]}')


Climate/dynamic cols (6): ['env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi']
Static cols          (9):  ['env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem']
Species cols         (173):   first 5 = ['Abaeis nicippe', 'Aglais milberti', 'Amblyscirtes hegon', 'Amblyscirtes vialis', 'Anartia fatima']


## 2. Build Split Indices and Data Arrays

In [ ]:
# ── Identical split logic to CISO butterfly notebook ─────────────────────────
train_split = np.array(butterfly_splits['train'])
val_split   = np.array(butterfly_splits['val'])
test_split  = np.array(butterfly_splits['test'])

assert train_split.max() < len(butterflies), 'Train index out of bounds'
assert val_split.max()   < len(butterflies), 'Val index out of bounds'
assert test_split.max()  < len(butterflies), 'Test index out of bounds'
print(f'train={len(train_split):,}  val={len(val_split):,}  test={len(test_split):,}')
print('All indices in bounds.')


train=13,825  val=950  test=2,302
All indices in bounds.


In [ ]:
# ── Filter species: >=100 occurrences (identical threshold to CISO notebook) ─
targets_full   = butterflies[species_cols].to_numpy().astype(np.float32)
species_counts = targets_full.sum(axis=0)
keep           = species_counts >= 100
targets        = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f'Species before filtering: {len(species_cols):,}')
print(f'Species after  filtering: {len(species_cols_filtered):,}')

# ── Tabular env features ──────────────────────────────────────────────────────
tabular_x = butterflies[env_cols].to_numpy().astype(np.float32)

# ── SatCLIP embeddings: not available, use zeros (masked out during training) ─
# MaskSDM's extra_masking will mask these just like real predictors.
N_SATCLIP = 256
satclip_embeddings = np.zeros((len(butterflies), N_SATCLIP), dtype=np.float32)

# ── Build data dict in MaskSDM format ────────────────────────────────────────
data = {
    'tabular_x':          tabular_x,
    'y':                  targets,
    'satclip_embeddings': satclip_embeddings,
}

# ── Per-split arrays ──────────────────────────────────────────────────────────
data['x_train'] = tabular_x[train_split]
data['x_val']   = tabular_x[val_split]
data['x_test']  = tabular_x[test_split]
data['y_train'] = targets[train_split]
data['y_val']   = targets[val_split]
data['y_test']  = targets[test_split]
data['satclip_embeddings_train'] = satclip_embeddings[train_split]
data['satclip_embeddings_val']   = satclip_embeddings[val_split]
data['satclip_embeddings_test']  = satclip_embeddings[test_split]

# ── Normalise using train statistics only ─────────────────────────────────────
train_mean = np.nanmean(data['x_train'], axis=0)
train_std  = np.nanstd(data['x_train'],  axis=0)
data['x_train'] = (data['x_train'] - train_mean) / (train_std + 1e-4)
data['x_val']   = (data['x_val']   - train_mean) / (train_std + 1e-4)
data['x_test']  = (data['x_test']  - train_mean) / (train_std + 1e-4)

print(f'x_train: {data["x_train"].shape}')
print(f'x_val:   {data["x_val"].shape}')
print(f'x_test:  {data["x_test"].shape}')


Species before filtering: 173
Species after  filtering: 171
x_train: (13825, 15)
x_val:   (950, 15)
x_test:  (2302, 15)


## 3. Verify Integrity

In [ ]:
n_features = data['x_train'].shape[1]
n_species  = data['y_train'].shape[1]

# Species that appear in all three splits — used for evaluation
indices_evaluated = np.intersect1d(
    np.intersect1d(
        data['y_train'].sum(axis=0).nonzero()[0],
        data['y_val'].sum(axis=0).nonzero()[0]
    ),
    data['y_test'].sum(axis=0).nonzero()[0]
).tolist()

print(f'n_features:        {n_features}')
print(f'n_species:         {n_species}')
print(f'species_evaluated: {len(indices_evaluated)}')
print(f'train rows:        {len(data["x_train"]):,}')
print(f'val rows:          {len(data["x_val"]):,}')
print(f'test rows:         {len(data["x_test"]):,}')
print('\nAll checks passed.')

n_features:        15
n_species:         171
species_evaluated: 144
train rows:        13,825
val rows:          950
test rows:         2,302

All checks passed.


## 4. Build Config and Train

In [ ]:
random_seed = 1337
seed_everything(random_seed)
torch.set_default_device(device)

# ── Edit these ────────────────────────────────────────────────────────────────
MAX_EPOCHS = 1000
BATCH_SIZE = 256
SAVE_DIR   = f'{CKPT_DIR}/masksdm_butterfly_1337'
# ─────────────────────────────────────────────────────────────────────────────

config = {
    'device':                    device,
    'seed':                      random_seed,
    # 'dataset' is a SplotDataset/GeoPlantDataset selector in MaskSDM's
    # data_helpers.py:88 — keep 'splot' to use the generic SplotDataset wrapper.
    # This does NOT mean we are using sPlot data.
    'dataset':                   'splot',
    'n_features':                n_features,
    'n_species':                 n_species,
    'n_samples_train':           len(data['y_train']),
    'n_samples_val':             len(data['y_val']),
    'n_samples_test':            len(data['y_test']),
    'indices_evaluated_species': indices_evaluated,
    'n_evaluated_species':       len(indices_evaluated),
    # satclip=False: zeros are always fully masked so they carry no signal
    'satclip':                   False,
    'model':                     'FTTransformer',
    'd_hidden':                  192,
    'n_heads':                   8,
    'n_blocks':                  7,
    'dropout':                   0.1,
    'd_out':                     n_species,
    'epochs':                    MAX_EPOCHS,
    'batch_size':                BATCH_SIZE,
    'batch_size_eval':           4096,
    'loss':                      'weighted',
    'species_weights':           torch.tensor(
                                     len(data['y_train']) / (data['y_train'].sum(axis=0) + 1e-5),
                                     dtype=torch.float32
                                 ).to(device),
    'optimizer':                 'AdamW',
    'scheduler_free':            True,
    'lr':                        0.001,
    'weight_decay':              0.01,
    'warmup_steps':              1000,
    'masking':                   True,
    'extra_masking':             True,
    'save_dir':                  SAVE_DIR,
    'use_wandb':                 False,
    'wandb_init':                {},
}

print(f'n_features:  {n_features}')
print(f'n_species:   {n_species}')
print(f'max_epochs:  {MAX_EPOCHS}')
print(f'save_dir:    {SAVE_DIR}')
print(f'device:      {device}')

train(config, data)


n_features:  15
n_species:   171
max_epochs:  1000
save_dir:    /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly
device:      cuda
Training model...
Epoch 1, val AUC: 0.516669
Epoch 2, val AUC: 0.543947
Epoch 3, val AUC: 0.563528
Epoch 4, val AUC: 0.573224
Epoch 5, val AUC: 0.587354
Epoch 6, val AUC: 0.655676
Epoch 7, val AUC: 0.712324
Epoch 8, val AUC: 0.728474
Epoch 9, val AUC: 0.743826
Epoch 10, val AUC: 0.751292
Epoch 11, val AUC: 0.753073
Epoch 12, val AUC: 0.763285
Epoch 13, val AUC: 0.772980
Epoch 14, val AUC: 0.776653
Epoch 15, val AUC: 0.782493
Epoch 16, val AUC: 0.787018
Epoch 17, val AUC: 0.780280
Epoch 18, val AUC: 0.779508
Epoch 19, val AUC: 0.784116
Epoch 20, val AUC: 0.790001
Epoch 21, val AUC: 0.794559
Epoch 22, val AUC: 0.795682
Epoch 23, val AUC: 0.795868
Epoch 24, val AUC: 0.792179
Epoch 25, val AUC: 0.795995
Epoch 26, val AUC: 0.799588
Epoch 27, val AUC: 0.800601
Epoch 28, val AUC: 0.800956
Epoch 29, val AUC: 0.802278
Epoch 30, val AUC: 0.801769
Epoch 31, va

FTTransformer(
  (feature_tokenizer): FeatureTokenizer(
    (periodic_embeddings): PeriodicEmbeddings(
      (periodic): _Periodic()
      (linear): _NLinear()
      (activation): ReLU()
    )
  )
  (satclip_projection): Linear(in_features=256, out_features=192, bias=True)
  (transformer_encoder): TransformerEncoder(
    (blocks): ModuleList(
      (0-6): 7 x ModuleDict(
        (attention_normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (attention): MultiheadAttention(
          (W_q): Linear(in_features=192, out_features=192, bias=True)
          (W_k): Linear(in_features=192, out_features=192, bias=True)
          (W_v): Linear(in_features=192, out_features=192, bias=True)
          (W_out): Linear(in_features=192, out_features=192, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (attention_residual_dropout): Dropout(p=0.1, inplace=False)
        (ffn_normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  

In [ ]:
import re

#BEST_EPOCH = S  # replace with the epoch number you found

#CHECKPOINT_PATH = f"{SAVE_DIR}/epoch_{BEST_EPOCH}.pt"

#print(f"Checkpoint: {CHECKPOINT_PATH}")


log = """Epoch 1, val AUC: 0.516669
Epoch 2, val AUC: 0.543947
Epoch 3, val AUC: 0.563528
Epoch 4, val AUC: 0.573224
Epoch 5, val AUC: 0.587354
Epoch 6, val AUC: 0.655676
Epoch 7, val AUC: 0.712324
Epoch 8, val AUC: 0.728474
Epoch 9, val AUC: 0.743826
Epoch 10, val AUC: 0.751292
Epoch 11, val AUC: 0.753073
Epoch 12, val AUC: 0.763285
Epoch 13, val AUC: 0.772980
Epoch 14, val AUC: 0.776653
Epoch 15, val AUC: 0.782493
Epoch 16, val AUC: 0.787018
Epoch 17, val AUC: 0.780280
Epoch 18, val AUC: 0.779508
Epoch 19, val AUC: 0.784116
Epoch 20, val AUC: 0.790001
Epoch 21, val AUC: 0.794559
Epoch 22, val AUC: 0.795682
Epoch 23, val AUC: 0.795868
Epoch 24, val AUC: 0.792179
Epoch 25, val AUC: 0.795995
Epoch 26, val AUC: 0.799588
Epoch 27, val AUC: 0.800601
Epoch 28, val AUC: 0.800956
Epoch 29, val AUC: 0.802278
Epoch 30, val AUC: 0.801769
Epoch 31, val AUC: 0.801396
Epoch 32, val AUC: 0.800561
Epoch 33, val AUC: 0.800574
Epoch 34, val AUC: 0.801141
Epoch 35, val AUC: 0.802957
Epoch 36, val AUC: 0.804718
Epoch 37, val AUC: 0.806321
Epoch 38, val AUC: 0.807926
Epoch 39, val AUC: 0.809891
Epoch 40, val AUC: 0.810809
Epoch 41, val AUC: 0.812233
Epoch 42, val AUC: 0.812937
Epoch 43, val AUC: 0.812309
Epoch 44, val AUC: 0.813354
Epoch 45, val AUC: 0.814346
Epoch 46, val AUC: 0.815435
Epoch 47, val AUC: 0.816663
Epoch 48, val AUC: 0.818559
Epoch 49, val AUC: 0.820305
Epoch 50, val AUC: 0.821202
Epoch 51, val AUC: 0.819087
Epoch 52, val AUC: 0.816329
Epoch 53, val AUC: 0.815518
Epoch 54, val AUC: 0.815105
Epoch 55, val AUC: 0.815383
Epoch 56, val AUC: 0.815770
Epoch 57, val AUC: 0.816110
Epoch 58, val AUC: 0.816671
Epoch 59, val AUC: 0.817308
Epoch 60, val AUC: 0.817859
Epoch 61, val AUC: 0.818467
Epoch 62, val AUC: 0.818686
Epoch 63, val AUC: 0.818727
Epoch 64, val AUC: 0.818812
Epoch 65, val AUC: 0.819010
Epoch 66, val AUC: 0.819058
Epoch 67, val AUC: 0.819091
Epoch 68, val AUC: 0.819172
Epoch 69, val AUC: 0.819337
Epoch 70, val AUC: 0.819536
Epoch 71, val AUC: 0.819631
Epoch 72, val AUC: 0.819793
Epoch 73, val AUC: 0.819985
Epoch 74, val AUC: 0.820661
Epoch 75, val AUC: 0.821020
Epoch 76, val AUC: 0.821691
Epoch 77, val AUC: 0.822108
Epoch 78, val AUC: 0.822453
Epoch 79, val AUC: 0.822833
Epoch 80, val AUC: 0.823099
Epoch 81, val AUC: 0.823263
Epoch 82, val AUC: 0.823431
Epoch 83, val AUC: 0.823608
Epoch 84, val AUC: 0.823701
Epoch 85, val AUC: 0.824080
Epoch 86, val AUC: 0.824461
Epoch 87, val AUC: 0.824675
Epoch 88, val AUC: 0.824946
Epoch 89, val AUC: 0.825462
Epoch 90, val AUC: 0.825977
Epoch 91, val AUC: 0.826509
Epoch 92, val AUC: 0.827017
Epoch 93, val AUC: 0.827277
Epoch 94, val AUC: 0.827585
Epoch 95, val AUC: 0.827910
Epoch 96, val AUC: 0.828500
Epoch 97, val AUC: 0.828499
Epoch 98, val AUC: 0.828863
Epoch 99, val AUC: 0.829511
Epoch 100, val AUC: 0.829695
Epoch 101, val AUC: 0.830072
Epoch 102, val AUC: 0.830145
Epoch 103, val AUC: 0.830226
Epoch 104, val AUC: 0.830375
Epoch 105, val AUC: 0.830385
Epoch 106, val AUC: 0.830539
Epoch 107, val AUC: 0.830842
Epoch 108, val AUC: 0.830913
Epoch 109, val AUC: 0.831224
Epoch 110, val AUC: 0.831557
Epoch 111, val AUC: 0.831742
Epoch 112, val AUC: 0.832161
Epoch 113, val AUC: 0.832533
Epoch 114, val AUC: 0.832893
Epoch 115, val AUC: 0.833258
Epoch 116, val AUC: 0.833600
Epoch 117, val AUC: 0.833889
Epoch 118, val AUC: 0.834114
Epoch 119, val AUC: 0.834128
Epoch 120, val AUC: 0.834367
Epoch 121, val AUC: 0.834694
Epoch 122, val AUC: 0.834993
Epoch 123, val AUC: 0.835276
Epoch 124, val AUC: 0.835640
Epoch 125, val AUC: 0.835719
Epoch 126, val AUC: 0.835899
Epoch 127, val AUC: 0.836286
Epoch 128, val AUC: 0.836204
Epoch 129, val AUC: 0.836234
Epoch 130, val AUC: 0.836311
Epoch 131, val AUC: 0.836586
Epoch 132, val AUC: 0.836789
Epoch 133, val AUC: 0.836905
Epoch 134, val AUC: 0.837029
Epoch 135, val AUC: 0.837221
Epoch 136, val AUC: 0.837289
Epoch 137, val AUC: 0.837187
Epoch 138, val AUC: 0.837140
Epoch 139, val AUC: 0.836807
Epoch 140, val AUC: 0.836612
Epoch 141, val AUC: 0.836494
Epoch 142, val AUC: 0.836572
Epoch 143, val AUC: 0.836680
Epoch 144, val AUC: 0.836905
Epoch 145, val AUC: 0.837249
Epoch 146, val AUC: 0.837421
Epoch 147, val AUC: 0.837626
Epoch 148, val AUC: 0.837774
Epoch 149, val AUC: 0.838097
Epoch 150, val AUC: 0.838272
Epoch 151, val AUC: 0.838590
Epoch 152, val AUC: 0.838732
Epoch 153, val AUC: 0.838727
Epoch 154, val AUC: 0.838846
Epoch 155, val AUC: 0.838892
Epoch 156, val AUC: 0.839013
Epoch 157, val AUC: 0.838890
Epoch 158, val AUC: 0.839065
Epoch 159, val AUC: 0.839193
Epoch 160, val AUC: 0.839277
Epoch 161, val AUC: 0.839752
Epoch 162, val AUC: 0.840072
Epoch 163, val AUC: 0.840270
Epoch 164, val AUC: 0.840373
Epoch 165, val AUC: 0.840376
Epoch 166, val AUC: 0.840345
Epoch 167, val AUC: 0.840140
Epoch 168, val AUC: 0.840098
Epoch 169, val AUC: 0.840073
Epoch 170, val AUC: 0.839917
Epoch 171, val AUC: 0.839615
Epoch 172, val AUC: 0.839599
Epoch 173, val AUC: 0.839580
Epoch 174, val AUC: 0.839642
Epoch 175, val AUC: 0.839558
Epoch 176, val AUC: 0.839575
Epoch 177, val AUC: 0.839578
Epoch 178, val AUC: 0.839548
Epoch 179, val AUC: 0.839624
Epoch 180, val AUC: 0.839557
Epoch 181, val AUC: 0.839604
Epoch 182, val AUC: 0.839629
Epoch 183, val AUC: 0.839475
Epoch 184, val AUC: 0.839411
Epoch 185, val AUC: 0.839288
Epoch 186, val AUC: 0.839222
Epoch 187, val AUC: 0.839093
Epoch 188, val AUC: 0.839001
Epoch 189, val AUC: 0.838904
Epoch 190, val AUC: 0.838781
Epoch 191, val AUC: 0.838705
Epoch 192, val AUC: 0.838665
Epoch 193, val AUC: 0.838884
Epoch 194, val AUC: 0.839020
Epoch 195, val AUC: 0.839028
Epoch 196, val AUC: 0.839050
Epoch 197, val AUC: 0.838909
Epoch 198, val AUC: 0.838768
Epoch 199, val AUC: 0.838602
Epoch 200, val AUC: 0.838417
Epoch 201, val AUC: 0.838341
Epoch 202, val AUC: 0.838215
Epoch 203, val AUC: 0.837934
Epoch 204, val AUC: 0.837627
Epoch 205, val AUC: 0.837522
Epoch 206, val AUC: 0.837413
Epoch 207, val AUC: 0.837310
Epoch 208, val AUC: 0.837199
Epoch 209, val AUC: 0.837282
Epoch 210, val AUC: 0.837267
Epoch 211, val AUC: 0.837241
Epoch 212, val AUC: 0.837207
Epoch 213, val AUC: 0.837230
Epoch 214, val AUC: 0.837196
Epoch 215, val AUC: 0.837127
Epoch 216, val AUC: 0.837145
Epoch 217, val AUC: 0.837248
Epoch 218, val AUC: 0.837347
Epoch 219, val AUC: 0.837386
Epoch 220, val AUC: 0.837426
Epoch 221, val AUC: 0.837444
Epoch 222, val AUC: 0.837530
Epoch 223, val AUC: 0.837625
Epoch 224, val AUC: 0.838480
Epoch 225, val AUC: 0.839401
Epoch 226, val AUC: 0.839967
Epoch 227, val AUC: 0.840276
Epoch 228, val AUC: 0.840396
Epoch 229, val AUC: 0.840673
Epoch 230, val AUC: 0.840906
Epoch 231, val AUC: 0.840984
Epoch 232, val AUC: 0.841072
Epoch 233, val AUC: 0.841014
Epoch 234, val AUC: 0.840831
Epoch 235, val AUC: 0.840629
Epoch 236, val AUC: 0.840543
Epoch 237, val AUC: 0.840338
Epoch 238, val AUC: 0.840229
Epoch 239, val AUC: 0.840094
Epoch 240, val AUC: 0.839979
Epoch 241, val AUC: 0.839947
Epoch 242, val AUC: 0.839863
Epoch 243, val AUC: 0.839690
Epoch 244, val AUC: 0.839575
Epoch 245, val AUC: 0.839528
Epoch 246, val AUC: 0.839472
Epoch 247, val AUC: 0.839423
Epoch 248, val AUC: 0.839298
Epoch 249, val AUC: 0.839072
Epoch 250, val AUC: 0.839046
Epoch 251, val AUC: 0.839061
Epoch 252, val AUC: 0.838932
Epoch 253, val AUC: 0.838778
Epoch 254, val AUC: 0.838579
Epoch 255, val AUC: 0.838405
Epoch 256, val AUC: 0.838281
Epoch 257, val AUC: 0.838188
Epoch 258, val AUC: 0.837968
Epoch 259, val AUC: 0.837808
Epoch 260, val AUC: 0.837665
Epoch 261, val AUC: 0.837569
Epoch 262, val AUC: 0.837595
Epoch 263, val AUC: 0.837610
Epoch 264, val AUC: 0.837428
Epoch 265, val AUC: 0.837317
Epoch 266, val AUC: 0.837186
Epoch 267, val AUC: 0.837082
Epoch 268, val AUC: 0.837081
Epoch 269, val AUC: 0.836961
Epoch 270, val AUC: 0.836949
Epoch 271, val AUC: 0.836981
Epoch 272, val AUC: 0.836904
Epoch 273, val AUC: 0.836906
Epoch 274, val AUC: 0.836845
Epoch 275, val AUC: 0.836676
Epoch 276, val AUC: 0.836602
Epoch 277, val AUC: 0.836608
Epoch 278, val AUC: 0.836604
Epoch 279, val AUC: 0.836657
Epoch 280, val AUC: 0.836828
Epoch 281, val AUC: 0.836874
Epoch 282, val AUC: 0.836936
Epoch 283, val AUC: 0.836964
Epoch 284, val AUC: 0.837160
Epoch 285, val AUC: 0.837068
Epoch 286, val AUC: 0.836971
Epoch 287, val AUC: 0.836991
Epoch 288, val AUC: 0.836906
Epoch 289, val AUC: 0.836838
Epoch 290, val AUC: 0.836851
Epoch 291, val AUC: 0.836869
Epoch 292, val AUC: 0.836784
Epoch 293, val AUC: 0.836645
Epoch 294, val AUC: 0.836503
Epoch 295, val AUC: 0.836417
Epoch 296, val AUC: 0.836452
Epoch 297, val AUC: 0.836485
Epoch 298, val AUC: 0.836475
Epoch 299, val AUC: 0.836523
Epoch 300, val AUC: 0.836610
Epoch 301, val AUC: 0.836588
Epoch 302, val AUC: 0.836590
Epoch 303, val AUC: 0.836534
Epoch 304, val AUC: 0.836550
Epoch 305, val AUC: 0.836515
Epoch 306, val AUC: 0.836609
Epoch 307, val AUC: 0.836669
Epoch 308, val AUC: 0.836714
Epoch 309, val AUC: 0.836707
Epoch 310, val AUC: 0.836698
Epoch 311, val AUC: 0.836622
Epoch 312, val AUC: 0.836592
Epoch 313, val AUC: 0.836559
Epoch 314, val AUC: 0.836552
Epoch 315, val AUC: 0.836469
Epoch 316, val AUC: 0.836490
Epoch 317, val AUC: 0.836472
Epoch 318, val AUC: 0.836524
Epoch 319, val AUC: 0.836541
Epoch 320, val AUC: 0.836565
Epoch 321, val AUC: 0.836547
Epoch 322, val AUC: 0.836508
Epoch 323, val AUC: 0.836522
Epoch 324, val AUC: 0.836521
Epoch 325, val AUC: 0.836528
Epoch 326, val AUC: 0.836475
Epoch 327, val AUC: 0.836548
Epoch 328, val AUC: 0.836619
Epoch 329, val AUC: 0.836678
Epoch 330, val AUC: 0.836681
Epoch 331, val AUC: 0.836625
Epoch 332, val AUC: 0.836637
Epoch 333, val AUC: 0.836586
Epoch 334, val AUC: 0.836483
Epoch 335, val AUC: 0.836517
Epoch 336, val AUC: 0.836507
Epoch 337, val AUC: 0.836449
Epoch 338, val AUC: 0.836456
Epoch 339, val AUC: 0.836423
Epoch 340, val AUC: 0.836434
Epoch 341, val AUC: 0.836438
Epoch 342, val AUC: 0.836458
Epoch 343, val AUC: 0.836453
Epoch 344, val AUC: 0.836502
Epoch 345, val AUC: 0.836485
Epoch 346, val AUC: 0.836417
Epoch 347, val AUC: 0.836421
Epoch 348, val AUC: 0.836405
Epoch 349, val AUC: 0.836330
Epoch 350, val AUC: 0.836307
Epoch 351, val AUC: 0.836384
Epoch 352, val AUC: 0.836435
Epoch 353, val AUC: 0.836554
Epoch 354, val AUC: 0.836603
Epoch 355, val AUC: 0.836696
Epoch 356, val AUC: 0.836734
Epoch 357, val AUC: 0.836753
Epoch 358, val AUC: 0.836803
Epoch 359, val AUC: 0.836885
Epoch 360, val AUC: 0.836919
Epoch 361, val AUC: 0.836958
Epoch 362, val AUC: 0.836970
Epoch 363, val AUC: 0.837019
Epoch 364, val AUC: 0.837200
Epoch 365, val AUC: 0.837351
Epoch 366, val AUC: 0.837451
Epoch 367, val AUC: 0.837544
Epoch 368, val AUC: 0.837612
Epoch 369, val AUC: 0.837714
Epoch 370, val AUC: 0.837828
Epoch 371, val AUC: 0.838009
Epoch 372, val AUC: 0.838083
Epoch 373, val AUC: 0.838182
Epoch 374, val AUC: 0.838316
Epoch 375, val AUC: 0.838526
Epoch 376, val AUC: 0.838686
Epoch 377, val AUC: 0.838840
Epoch 378, val AUC: 0.839081
Epoch 379, val AUC: 0.839179
Epoch 380, val AUC: 0.839343
Epoch 381, val AUC: 0.839493
Epoch 382, val AUC: 0.839490
Epoch 383, val AUC: 0.839548
Epoch 384, val AUC: 0.839633
Epoch 385, val AUC: 0.839764
Epoch 386, val AUC: 0.839892
Epoch 387, val AUC: 0.839929
Epoch 388, val AUC: 0.840012
Epoch 389, val AUC: 0.840131
Epoch 390, val AUC: 0.840206
Epoch 391, val AUC: 0.840388
Epoch 392, val AUC: 0.840619
Epoch 393, val AUC: 0.840787
Epoch 394, val AUC: 0.840917
Epoch 395, val AUC: 0.841105
Epoch 396, val AUC: 0.841291
Epoch 397, val AUC: 0.841460
Epoch 398, val AUC: 0.841595
Epoch 399, val AUC: 0.841756
Epoch 400, val AUC: 0.841887
Epoch 401, val AUC: 0.842053
Epoch 402, val AUC: 0.842162
Epoch 403, val AUC: 0.842292
Epoch 404, val AUC: 0.842390
Epoch 405, val AUC: 0.842492
Epoch 406, val AUC: 0.842577
Epoch 407, val AUC: 0.842754
Epoch 408, val AUC: 0.842914
Epoch 409, val AUC: 0.843117
Epoch 410, val AUC: 0.843200
Epoch 411, val AUC: 0.843354
Epoch 412, val AUC: 0.843443
Epoch 413, val AUC: 0.843470
Epoch 414, val AUC: 0.843522
Epoch 415, val AUC: 0.843598
Epoch 416, val AUC: 0.843626
Epoch 417, val AUC: 0.843719
Epoch 418, val AUC: 0.843806
Epoch 419, val AUC: 0.843877
Epoch 420, val AUC: 0.843974
Epoch 421, val AUC: 0.844024
Epoch 422, val AUC: 0.844097
Epoch 423, val AUC: 0.844132
Epoch 424, val AUC: 0.844185
Epoch 425, val AUC: 0.844233
Epoch 426, val AUC: 0.844330
Epoch 427, val AUC: 0.844402
Epoch 428, val AUC: 0.844437
Epoch 429, val AUC: 0.844436
Epoch 430, val AUC: 0.844484
Epoch 431, val AUC: 0.844492
Epoch 432, val AUC: 0.844516
Epoch 433, val AUC: 0.844533
Epoch 434, val AUC: 0.844622
Epoch 435, val AUC: 0.844595
Epoch 436, val AUC: 0.844601
Epoch 437, val AUC: 0.844573
Epoch 438, val AUC: 0.844537
Epoch 439, val AUC: 0.844533
Epoch 440, val AUC: 0.844599
Epoch 441, val AUC: 0.844601
Epoch 442, val AUC: 0.844599
Epoch 443, val AUC: 0.844600
Epoch 444, val AUC: 0.844656
Epoch 445, val AUC: 0.844607
Epoch 446, val AUC: 0.844630
Epoch 447, val AUC: 0.844643
Epoch 448, val AUC: 0.844583
Epoch 449, val AUC: 0.844584
Epoch 450, val AUC: 0.844522
Epoch 451, val AUC: 0.844547
Epoch 452, val AUC: 0.844486
Epoch 453, val AUC: 0.844466
Epoch 454, val AUC: 0.844467
Epoch 455, val AUC: 0.844481
Epoch 456, val AUC: 0.844497
Epoch 457, val AUC: 0.844439
Epoch 458, val AUC: 0.844421
Epoch 459, val AUC: 0.844395
Epoch 460, val AUC: 0.844410
Epoch 461, val AUC: 0.844358
Epoch 462, val AUC: 0.844375
Epoch 463, val AUC: 0.844411
Epoch 464, val AUC: 0.844440
Epoch 465, val AUC: 0.844438
Epoch 466, val AUC: 0.844459
Epoch 467, val AUC: 0.844418
Epoch 468, val AUC: 0.844402
Epoch 469, val AUC: 0.844457
Epoch 470, val AUC: 0.844499
Epoch 471, val AUC: 0.844528
Epoch 472, val AUC: 0.844448
Epoch 473, val AUC: 0.844419
Epoch 474, val AUC: 0.844390
Epoch 475, val AUC: 0.844423
Epoch 476, val AUC: 0.844381
Epoch 477, val AUC: 0.844354
Epoch 478, val AUC: 0.844340
Epoch 479, val AUC: 0.844309
Epoch 480, val AUC: 0.844296
Epoch 481, val AUC: 0.844231
Epoch 482, val AUC: 0.844178
Epoch 483, val AUC: 0.844201
Epoch 484, val AUC: 0.844145
Epoch 485, val AUC: 0.844128
Epoch 486, val AUC: 0.844126
Epoch 487, val AUC: 0.844141
Epoch 488, val AUC: 0.844110
Epoch 489, val AUC: 0.844042
Epoch 490, val AUC: 0.843982
Epoch 491, val AUC: 0.843938
Epoch 492, val AUC: 0.843903
Epoch 493, val AUC: 0.843911
Epoch 494, val AUC: 0.843902
Epoch 495, val AUC: 0.843889
Epoch 496, val AUC: 0.843887
Epoch 497, val AUC: 0.843883
Epoch 498, val AUC: 0.843890
Epoch 499, val AUC: 0.843862
Epoch 500, val AUC: 0.843871
Epoch 501, val AUC: 0.843896
Epoch 502, val AUC: 0.843888
Epoch 503, val AUC: 0.843919
Epoch 504, val AUC: 0.843909
Epoch 505, val AUC: 0.843918
Epoch 506, val AUC: 0.843898
Epoch 507, val AUC: 0.843920
Epoch 508, val AUC: 0.843950
Epoch 509, val AUC: 0.844009
Epoch 510, val AUC: 0.844072
Epoch 511, val AUC: 0.844034
Epoch 512, val AUC: 0.844023
Epoch 513, val AUC: 0.844033
Epoch 514, val AUC: 0.843958
Epoch 515, val AUC: 0.843982
Epoch 516, val AUC: 0.843993
Epoch 517, val AUC: 0.843946
Epoch 518, val AUC: 0.843938
Epoch 519, val AUC: 0.843982
Epoch 520, val AUC: 0.843991
Epoch 521, val AUC: 0.843989
Epoch 522, val AUC: 0.843963
Epoch 523, val AUC: 0.843886
Epoch 524, val AUC: 0.843865
Epoch 525, val AUC: 0.843869
Epoch 526, val AUC: 0.843875
Epoch 527, val AUC: 0.843779
Epoch 528, val AUC: 0.843764
Epoch 529, val AUC: 0.843718
Epoch 530, val AUC: 0.843711
Epoch 531, val AUC: 0.843675
Epoch 532, val AUC: 0.843668
Epoch 533, val AUC: 0.843672
Epoch 534, val AUC: 0.843631
Epoch 535, val AUC: 0.843592
Epoch 536, val AUC: 0.843560
Epoch 537, val AUC: 0.843566
Epoch 538, val AUC: 0.843529
Epoch 539, val AUC: 0.843556
Epoch 540, val AUC: 0.843589
Epoch 541, val AUC: 0.843607
Epoch 542, val AUC: 0.843617
Epoch 543, val AUC: 0.843567
Epoch 544, val AUC: 0.843541
Epoch 545, val AUC: 0.843570
Epoch 546, val AUC: 0.843518
Epoch 547, val AUC: 0.843503
Epoch 548, val AUC: 0.843491
Epoch 549, val AUC: 0.843519
Epoch 550, val AUC: 0.843493
Epoch 551, val AUC: 0.843512
Epoch 552, val AUC: 0.843486
Epoch 553, val AUC: 0.843492
Epoch 554, val AUC: 0.843496
Epoch 555, val AUC: 0.843486
Epoch 556, val AUC: 0.843482
Epoch 557, val AUC: 0.843469
Epoch 558, val AUC: 0.843354
Epoch 559, val AUC: 0.843248
Epoch 560, val AUC: 0.843187
Epoch 561, val AUC: 0.843157
Epoch 562, val AUC: 0.843117
Epoch 563, val AUC: 0.843082
Epoch 564, val AUC: 0.843040
Epoch 565, val AUC: 0.842973
Epoch 566, val AUC: 0.842884
Epoch 567, val AUC: 0.842814
Epoch 568, val AUC: 0.842742
Epoch 569, val AUC: 0.842650
Epoch 570, val AUC: 0.842619
Epoch 571, val AUC: 0.842592
Epoch 572, val AUC: 0.842516
Epoch 573, val AUC: 0.842464
Epoch 574, val AUC: 0.842446
Epoch 575, val AUC: 0.842386
Epoch 576, val AUC: 0.842351
Epoch 577, val AUC: 0.842300
Epoch 578, val AUC: 0.842270
Epoch 579, val AUC: 0.842245
Epoch 580, val AUC: 0.842288
Epoch 581, val AUC: 0.842375
Epoch 582, val AUC: 0.842478
Epoch 583, val AUC: 0.842498
Epoch 584, val AUC: 0.842486
Epoch 585, val AUC: 0.842499
Epoch 586, val AUC: 0.842530
Epoch 587, val AUC: 0.842533
Epoch 588, val AUC: 0.842568
Epoch 589, val AUC: 0.842548
Epoch 590, val AUC: 0.842574
Epoch 591, val AUC: 0.842579
Epoch 592, val AUC: 0.842621
Epoch 593, val AUC: 0.842622
Epoch 594, val AUC: 0.842610
Epoch 595, val AUC: 0.842624
Epoch 596, val AUC: 0.842675
Epoch 597, val AUC: 0.842767
Epoch 598, val AUC: 0.842743
Epoch 599, val AUC: 0.842806
Epoch 600, val AUC: 0.842808
Epoch 601, val AUC: 0.842842
Epoch 602, val AUC: 0.842882
Epoch 603, val AUC: 0.842938
Epoch 604, val AUC: 0.843022
Epoch 605, val AUC: 0.843058
Epoch 606, val AUC: 0.843043
Epoch 607, val AUC: 0.843088
Epoch 608, val AUC: 0.843144
Epoch 609, val AUC: 0.843182
Epoch 610, val AUC: 0.843162
Epoch 611, val AUC: 0.843188
Epoch 612, val AUC: 0.843149
Epoch 613, val AUC: 0.843189
Epoch 614, val AUC: 0.843310
Epoch 615, val AUC: 0.843403
Epoch 616, val AUC: 0.843439
Epoch 617, val AUC: 0.843537
Epoch 618, val AUC: 0.843574
Epoch 619, val AUC: 0.843663
Epoch 620, val AUC: 0.843757
Epoch 621, val AUC: 0.843816
Epoch 622, val AUC: 0.843900
Epoch 623, val AUC: 0.843989
Epoch 624, val AUC: 0.844038
Epoch 625, val AUC: 0.844105
Epoch 626, val AUC: 0.844175
Epoch 627, val AUC: 0.844244
Epoch 628, val AUC: 0.844332
Epoch 629, val AUC: 0.844402
Epoch 630, val AUC: 0.844500
Epoch 631, val AUC: 0.844556
Epoch 632, val AUC: 0.844618
Epoch 633, val AUC: 0.844626
Epoch 634, val AUC: 0.844641
Epoch 635, val AUC: 0.844709
Epoch 636, val AUC: 0.844744
Epoch 637, val AUC: 0.844799
Epoch 638, val AUC: 0.844747
Epoch 639, val AUC: 0.844736
Epoch 640, val AUC: 0.844729
Epoch 641, val AUC: 0.844695
Epoch 642, val AUC: 0.844674
Epoch 643, val AUC: 0.844709
Epoch 644, val AUC: 0.844738
Epoch 645, val AUC: 0.844737
Epoch 646, val AUC: 0.844767
Epoch 647, val AUC: 0.844775
Epoch 648, val AUC: 0.844770
Epoch 649, val AUC: 0.844778
Epoch 650, val AUC: 0.844814
Epoch 651, val AUC: 0.844821
Epoch 652, val AUC: 0.844844
Epoch 653, val AUC: 0.844907
Epoch 654, val AUC: 0.844965
Epoch 655, val AUC: 0.845015
Epoch 656, val AUC: 0.845033
Epoch 657, val AUC: 0.845137
Epoch 658, val AUC: 0.845168
Epoch 659, val AUC: 0.845229
Epoch 660, val AUC: 0.845267
Epoch 661, val AUC: 0.845389
Epoch 662, val AUC: 0.845424
Epoch 663, val AUC: 0.845511
Epoch 664, val AUC: 0.845596
Epoch 665, val AUC: 0.845665
Epoch 666, val AUC: 0.845721
Epoch 667, val AUC: 0.845753
Epoch 668, val AUC: 0.845850
Epoch 669, val AUC: 0.845914
Epoch 670, val AUC: 0.845978
Epoch 671, val AUC: 0.846049
Epoch 672, val AUC: 0.846167
Epoch 673, val AUC: 0.846250
Epoch 674, val AUC: 0.846294
Epoch 675, val AUC: 0.846331
Epoch 676, val AUC: 0.846397
Epoch 677, val AUC: 0.846431
Epoch 678, val AUC: 0.846460
Epoch 679, val AUC: 0.846496
Epoch 680, val AUC: 0.846544
Epoch 681, val AUC: 0.846564
Epoch 682, val AUC: 0.846626
Epoch 683, val AUC: 0.846657
Epoch 684, val AUC: 0.846728
Epoch 685, val AUC: 0.846700
Epoch 686, val AUC: 0.846713
Epoch 687, val AUC: 0.846729
Epoch 688, val AUC: 0.846706
Epoch 689, val AUC: 0.846681
Epoch 690, val AUC: 0.846627
Epoch 691, val AUC: 0.846586
Epoch 692, val AUC: 0.846587
Epoch 693, val AUC: 0.846621
Epoch 694, val AUC: 0.846671
Epoch 695, val AUC: 0.846732
Epoch 696, val AUC: 0.846758
Epoch 697, val AUC: 0.846804
Epoch 698, val AUC: 0.846830
Epoch 699, val AUC: 0.846819
Epoch 700, val AUC: 0.846840
Epoch 701, val AUC: 0.846847
Epoch 702, val AUC: 0.846823
Epoch 703, val AUC: 0.846850
Epoch 704, val AUC: 0.846804
Epoch 705, val AUC: 0.846810
Epoch 706, val AUC: 0.846774
Epoch 707, val AUC: 0.846800
Epoch 708, val AUC: 0.846768
Epoch 709, val AUC: 0.846751
Epoch 710, val AUC: 0.846754
Epoch 711, val AUC: 0.846774
Epoch 712, val AUC: 0.846823
Epoch 713, val AUC: 0.846878
Epoch 714, val AUC: 0.846909
Epoch 715, val AUC: 0.846934
Epoch 716, val AUC: 0.846939
Epoch 717, val AUC: 0.846945
Epoch 718, val AUC: 0.847057
Epoch 719, val AUC: 0.847098
Epoch 720, val AUC: 0.847071
Epoch 721, val AUC: 0.847082
Epoch 722, val AUC: 0.847136
Epoch 723, val AUC: 0.847140
Epoch 724, val AUC: 0.847079
Epoch 725, val AUC: 0.847037
Epoch 726, val AUC: 0.847046
Epoch 727, val AUC: 0.847007
Epoch 728, val AUC: 0.846941
Epoch 729, val AUC: 0.846966
Epoch 730, val AUC: 0.846955
Epoch 731, val AUC: 0.846957
Epoch 732, val AUC: 0.846917
Epoch 733, val AUC: 0.846893
Epoch 734, val AUC: 0.846837
Epoch 735, val AUC: 0.846987
Epoch 736, val AUC: 0.847114
Epoch 737, val AUC: 0.847183
Epoch 738, val AUC: 0.847256
Epoch 739, val AUC: 0.847364
Epoch 740, val AUC: 0.847448
Epoch 741, val AUC: 0.847479
Epoch 742, val AUC: 0.847640
Epoch 743, val AUC: 0.847743
Epoch 744, val AUC: 0.847832
Epoch 745, val AUC: 0.847932
Epoch 746, val AUC: 0.848020
Epoch 747, val AUC: 0.848074
Epoch 748, val AUC: 0.848141
Epoch 749, val AUC: 0.848174
Epoch 750, val AUC: 0.848241
Epoch 751, val AUC: 0.848267
Epoch 752, val AUC: 0.848224
Epoch 753, val AUC: 0.848252
Epoch 754, val AUC: 0.848312
Epoch 755, val AUC: 0.848336
Epoch 756, val AUC: 0.848330
Epoch 757, val AUC: 0.848329
Epoch 758, val AUC: 0.848338
Epoch 759, val AUC: 0.848333
Epoch 760, val AUC: 0.848255
Epoch 761, val AUC: 0.848220
Epoch 762, val AUC: 0.848304
Epoch 763, val AUC: 0.848354
Epoch 764, val AUC: 0.848353
Epoch 765, val AUC: 0.848345
Epoch 766, val AUC: 0.848394
Epoch 767, val AUC: 0.848400
Epoch 768, val AUC: 0.848451
Epoch 769, val AUC: 0.848455
Epoch 770, val AUC: 0.848508
Epoch 771, val AUC: 0.848603
Epoch 772, val AUC: 0.848639
Epoch 773, val AUC: 0.848649
Epoch 774, val AUC: 0.848620
Epoch 775, val AUC: 0.848616
Epoch 776, val AUC: 0.848631
Epoch 777, val AUC: 0.848648
Epoch 778, val AUC: 0.848644
Epoch 779, val AUC: 0.848613
Epoch 780, val AUC: 0.848597
Epoch 781, val AUC: 0.848576
Epoch 782, val AUC: 0.848543
Epoch 783, val AUC: 0.848523
Epoch 784, val AUC: 0.848486
Epoch 785, val AUC: 0.848456
Epoch 786, val AUC: 0.848386
Epoch 787, val AUC: 0.848306
Epoch 788, val AUC: 0.848307
Epoch 789, val AUC: 0.848276
Epoch 790, val AUC: 0.848266
Epoch 791, val AUC: 0.848276
Epoch 792, val AUC: 0.848263
Epoch 793, val AUC: 0.848251
Epoch 794, val AUC: 0.848237
Epoch 795, val AUC: 0.848123
Epoch 796, val AUC: 0.848072
Epoch 797, val AUC: 0.848039
Epoch 798, val AUC: 0.848053
Epoch 799, val AUC: 0.848021
Epoch 800, val AUC: 0.848002
Epoch 801, val AUC: 0.847981
Epoch 802, val AUC: 0.847998
Epoch 803, val AUC: 0.847945
Epoch 804, val AUC: 0.847910
Epoch 805, val AUC: 0.847868
Epoch 806, val AUC: 0.847817
Epoch 807, val AUC: 0.847798
Epoch 808, val AUC: 0.847723
Epoch 809, val AUC: 0.847711
Epoch 810, val AUC: 0.847690
Epoch 811, val AUC: 0.847666
Epoch 812, val AUC: 0.847657
Epoch 813, val AUC: 0.847637
Epoch 814, val AUC: 0.847597
Epoch 815, val AUC: 0.847531
Epoch 816, val AUC: 0.847533
Epoch 817, val AUC: 0.847529
Epoch 818, val AUC: 0.847499
Epoch 819, val AUC: 0.847485
Epoch 820, val AUC: 0.847481
Epoch 821, val AUC: 0.847440
Epoch 822, val AUC: 0.847408
Epoch 823, val AUC: 0.847415
Epoch 824, val AUC: 0.847396
Epoch 825, val AUC: 0.847383
Epoch 826, val AUC: 0.847375
Epoch 827, val AUC: 0.847413
Epoch 828, val AUC: 0.847424
Epoch 829, val AUC: 0.847436
Epoch 830, val AUC: 0.847416
Epoch 831, val AUC: 0.847371
Epoch 832, val AUC: 0.847311
Epoch 833, val AUC: 0.847303
Epoch 834, val AUC: 0.847292
Epoch 835, val AUC: 0.847333
Epoch 836, val AUC: 0.847345
Epoch 837, val AUC: 0.847296
Epoch 838, val AUC: 0.847273
Epoch 839, val AUC: 0.847220
Epoch 840, val AUC: 0.847151
Epoch 841, val AUC: 0.847158
Epoch 842, val AUC: 0.847146
Epoch 843, val AUC: 0.847120
Epoch 844, val AUC: 0.847078
Epoch 845, val AUC: 0.847077
Epoch 846, val AUC: 0.847116
Epoch 847, val AUC: 0.847110
Epoch 848, val AUC: 0.847151
Epoch 849, val AUC: 0.847157
Epoch 850, val AUC: 0.847127
Epoch 851, val AUC: 0.847104
Epoch 852, val AUC: 0.847113
Epoch 853, val AUC: 0.847106
Epoch 854, val AUC: 0.847041
Epoch 855, val AUC: 0.847028
Epoch 856, val AUC: 0.846998
Epoch 857, val AUC: 0.846980
Epoch 858, val AUC: 0.846946
Epoch 859, val AUC: 0.846911
Epoch 860, val AUC: 0.846901
Epoch 861, val AUC: 0.846867
Epoch 862, val AUC: 0.846860
Epoch 863, val AUC: 0.846856
Epoch 864, val AUC: 0.846852
Epoch 865, val AUC: 0.846859
Epoch 866, val AUC: 0.846820
Epoch 867, val AUC: 0.846763
Epoch 868, val AUC: 0.846748
Epoch 869, val AUC: 0.846714
Epoch 870, val AUC: 0.846678
Epoch 871, val AUC: 0.846695
Epoch 872, val AUC: 0.846663
Epoch 873, val AUC: 0.846649
Epoch 874, val AUC: 0.846610
Epoch 875, val AUC: 0.846560
Epoch 876, val AUC: 0.846527
Epoch 877, val AUC: 0.846544
Epoch 878, val AUC: 0.846513
Epoch 879, val AUC: 0.846497
Epoch 880, val AUC: 0.846480
Epoch 881, val AUC: 0.846432
Epoch 882, val AUC: 0.846426
Epoch 883, val AUC: 0.846385
Epoch 884, val AUC: 0.846378
Epoch 885, val AUC: 0.846345
Epoch 886, val AUC: 0.846306
Epoch 887, val AUC: 0.846282
Epoch 888, val AUC: 0.846250
Epoch 889, val AUC: 0.846200
Epoch 890, val AUC: 0.846174
Epoch 891, val AUC: 0.846184
Epoch 892, val AUC: 0.846171
Epoch 893, val AUC: 0.846148
Epoch 894, val AUC: 0.846141
Epoch 895, val AUC: 0.846125
Epoch 896, val AUC: 0.846105
Epoch 897, val AUC: 0.846063
Epoch 898, val AUC: 0.846047
Epoch 899, val AUC: 0.846040
Epoch 900, val AUC: 0.846046
Epoch 901, val AUC: 0.846061
Epoch 902, val AUC: 0.846076
Epoch 903, val AUC: 0.846080
Epoch 904, val AUC: 0.846071
Epoch 905, val AUC: 0.846088
Epoch 906, val AUC: 0.846122
Epoch 907, val AUC: 0.846153
Epoch 908, val AUC: 0.846161
Epoch 909, val AUC: 0.846152
Epoch 910, val AUC: 0.846196
Epoch 911, val AUC: 0.846257
Epoch 912, val AUC: 0.846259
Epoch 913, val AUC: 0.846286
Epoch 914, val AUC: 0.846305
Epoch 915, val AUC: 0.846360
Epoch 916, val AUC: 0.846439
Epoch 917, val AUC: 0.846456
Epoch 918, val AUC: 0.846432
Epoch 919, val AUC: 0.846425
Epoch 920, val AUC: 0.846458
Epoch 921, val AUC: 0.846480
Epoch 922, val AUC: 0.846469
Epoch 923, val AUC: 0.846472
Epoch 924, val AUC: 0.846428
Epoch 925, val AUC: 0.846413
Epoch 926, val AUC: 0.846440
Epoch 927, val AUC: 0.846422
Epoch 928, val AUC: 0.846407
Epoch 929, val AUC: 0.846390
Epoch 930, val AUC: 0.846380
Epoch 931, val AUC: 0.846325
Epoch 932, val AUC: 0.846312
Epoch 933, val AUC: 0.846280
Epoch 934, val AUC: 0.846241
Epoch 935, val AUC: 0.846218
Epoch 936, val AUC: 0.846206
Epoch 937, val AUC: 0.846219
Epoch 938, val AUC: 0.846230
Epoch 939, val AUC: 0.846232
Epoch 940, val AUC: 0.846180
Epoch 941, val AUC: 0.846180
Epoch 942, val AUC: 0.846161
Epoch 943, val AUC: 0.846184
Epoch 944, val AUC: 0.846187
Epoch 945, val AUC: 0.846197
Epoch 946, val AUC: 0.846205
Epoch 947, val AUC: 0.846172
Epoch 948, val AUC: 0.846167
Epoch 949, val AUC: 0.846163
Epoch 950, val AUC: 0.846199
Epoch 951, val AUC: 0.846205
Epoch 952, val AUC: 0.846173
Epoch 953, val AUC: 0.846194
Epoch 954, val AUC: 0.846203
Epoch 955, val AUC: 0.846164
Epoch 956, val AUC: 0.846194
Epoch 957, val AUC: 0.846197
Epoch 958, val AUC: 0.846197
Epoch 959, val AUC: 0.846229
Epoch 960, val AUC: 0.846218
Epoch 961, val AUC: 0.846211
Epoch 962, val AUC: 0.846173
Epoch 963, val AUC: 0.846191
Epoch 964, val AUC: 0.846184
Epoch 965, val AUC: 0.846144
Epoch 966, val AUC: 0.846116
Epoch 967, val AUC: 0.846114
Epoch 968, val AUC: 0.846061
Epoch 969, val AUC: 0.846027
Epoch 970, val AUC: 0.845988
Epoch 971, val AUC: 0.845950
Epoch 972, val AUC: 0.845886
Epoch 973, val AUC: 0.845912
Epoch 974, val AUC: 0.845869
Epoch 975, val AUC: 0.845866
Epoch 976, val AUC: 0.845824
Epoch 977, val AUC: 0.845807
Epoch 978, val AUC: 0.845811
Epoch 979, val AUC: 0.845822
Epoch 980, val AUC: 0.845842
Epoch 981, val AUC: 0.845857
Epoch 982, val AUC: 0.845847
Epoch 983, val AUC: 0.845840
Epoch 984, val AUC: 0.845830
Epoch 985, val AUC: 0.845837
Epoch 986, val AUC: 0.845825
Epoch 987, val AUC: 0.845785
Epoch 988, val AUC: 0.845820
Epoch 989, val AUC: 0.845791
Epoch 990, val AUC: 0.845767
Epoch 991, val AUC: 0.845735
Epoch 992, val AUC: 0.845727
Epoch 993, val AUC: 0.845757
Epoch 994, val AUC: 0.845747
Epoch 995, val AUC: 0.845797
Epoch 996, val AUC: 0.845820
Epoch 997, val AUC: 0.845797
Epoch 998, val AUC: 0.845775
Epoch 999, val AUC: 0.845823
Epoch 1000, val AUC: 0.845849"""

matches = [
    (int(m.group(1)), float(m.group(2)))
    for m in re.finditer(r"Epoch (\d+), val AUC: ([\d.]+)", log)
]

if not matches:
    raise ValueError("No lines matching 'Epoch <n>, val AUC: <value>' were found in log.")

epoch, auc = max(matches, key=lambda x: x[1])

CHECKPOINT_PATH = f"{SAVE_DIR}/epoch_{epoch}.pt"

print(f"Best epoch: {epoch}  (val AUC = {auc:.6f})")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Best epoch: 773  (val AUC = 0.848649)
Checkpoint: /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly/epoch_773.pt


## 5. Inference with 100% Masking (p=1.0)

All tabular predictors are masked — the model receives only the learned mask token for every input.
This matches STEM-LM p=1.0 (fully unconditioned).
MaskSDM is trained with masked data modelling so this is the setting it is designed to handle gracefully.

In [ ]:
model = get_model(config).to(device)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Model loaded.')

Model loaded.


In [ ]:
test_loader = DataLoader(
    get_torch_dataset(config, data['x_test'], data['y_test'], data['satclip_embeddings_test']),
    batch_size=config['batch_size_eval'],
    shuffle=False,
)

all_probs, all_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        x_batch, y_batch, satclip_emb = batch
        x_batch     = x_batch.to(device)
        satclip_emb = satclip_emb.to(device)

        # Match upstream MaskSDM evaluate(): keep all valid features (mask=1=kept).
        # SatCLIP stays masked because we did not train with SatCLIP embeddings.
        x_mask       = ~torch.isnan(x_batch).to(device)
        satclip_mask = torch.zeros(len(satclip_emb), dtype=torch.bool, device=device)

        logits = model(x_batch, satclip_emb, x_mask, satclip_mask)
        probs  = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

probs   = np.concatenate(all_probs,   axis=0)                    # (N_test, S)
targets = np.concatenate(all_targets, axis=0).astype(np.int64)   # (N_test, S)

print(f'probs:   {probs.shape}')
print(f'targets: {targets.shape}')

probs:   (2302, 171)
targets: (2302, 171)


## 6. STEM-LM Metrics

In [ ]:
# ── Identical metric primitives to CISO notebook (STEMLM_metric.py) ──────────

def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx   = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')

print('Metric functions defined.')

Metric functions defined.


In [ ]:
# At p=1.0 every feature is masked for every row, so we use all test rows.
# Per-split species filter: skip species with no presences (or no absences) in test
# — matches how the R baselines (Logistic / GAM / Maxnet) report n_species.
auc_roc_vals, auc_pr_vals, cbi_vals, brier_vals, ece_vals = [], [], [], [], []

for s in range(targets.shape[1]):
    y = targets[:, s].astype(np.int64)
    p = probs[:, s].astype(np.float64)
    if y.sum() == 0 or y.sum() == len(y):
        continue
    auc_roc_vals.append(_safe_auc_roc(y, p))
    auc_pr_vals.append(_safe_auc_pr(y, p))
    cbi_vals.append(_safe_cbi(y, p))
    brier_vals.append(_safe_brier(y, p))
    ece_vals.append(_safe_ece(y, p))

def _nanmean(v):   return float(np.nanmean([x for x in v if np.isfinite(x)])) if v else float('nan')
def _nanq(v, q):   vals=[x for x in v if np.isfinite(x)]; return float(np.quantile(vals,q)) if vals else float('nan')

summary = {
    'model':          'MaskSDM',
    'masking_p':      1.0,
    'eval_known_ratio': 0.0,
    'n_species':      len(auc_roc_vals),
    'mean_auc_roc':   _nanmean(auc_roc_vals),
    'auc_roc_q25':    _nanq(auc_roc_vals, 0.25),
    'auc_roc_q50':    _nanq(auc_roc_vals, 0.50),
    'auc_roc_q75':    _nanq(auc_roc_vals, 0.75),
    'mean_auc_pr':    _nanmean(auc_pr_vals),
    'mean_cbi':       _nanmean(cbi_vals),
    'mean_brier':     _nanmean(brier_vals),
    'mean_ece':       _nanmean(ece_vals),
}

cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species']

print('\n-- MaskSDM Benchmark (p=1.0, fully unconditioned) --')
for k in cols:
    v = summary[k]
    print(f'  {k:<20} {round(v, 4) if isinstance(v, float) else v}')


-- MaskSDM Benchmark (p=1.0, fully unconditioned) --
  mean_auc_roc         0.8225
  auc_roc_q25          0.7316
  auc_roc_q50          0.8324
  auc_roc_q75          0.93
  mean_auc_pr          0.1487
  mean_cbi             0.4894
  mean_brier           0.0869
  mean_ece             0.1228
  n_species            151


In [ ]:
import json as _json

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

out_csv  = f'{RESULTS_DIR}/masksdm_benchmark_summary.csv'
out_json = f'{RESULTS_DIR}/masksdm_benchmark_summary.json'

pd.DataFrame([summary]).set_index('masking_p').to_csv(out_csv)
with open(out_json, 'w') as f:
    _json.dump(summary, f, indent=2)

print(f'CSV  saved to {out_csv}')
print(f'JSON saved to {out_json}')

CSV  saved to /content/drive/MyDrive/CISO/eButterfly/results_masksdm_butterfly_20_epochs/masksdm_benchmark_summary.csv
JSON saved to /content/drive/MyDrive/CISO/eButterfly/results_masksdm_butterfly_20_epochs/masksdm_benchmark_summary.json
